In [1]:
# ---- Setup: Drive + code + wandb (run this first, every session) ----
REPO_URL = "https://github.com/ammara-asif/fyp.git"   # <-- EDIT ONCE
BRANCH = "main"

import os, subprocess, sys
from google.colab import drive, userdata

# Drive must be mounted from the notebook kernel (never from a !python subprocess).
drive.mount("/content/drive")

# Code: fresh clone each session (Colab disk is wiped on reset), pull if it already exists.
# For a private repo, add a Colab secret GITHUB_TOKEN (a fine-grained read-only token).
CODE = "/content/fyp"
url = REPO_URL
try:
    tok = userdata.get("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if os.path.isdir(CODE):
    subprocess.run(["git", "-C", CODE, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", "-b", BRANCH, url, CODE], check=True)
print(subprocess.check_output(["git", "-C", CODE, "log", "-1", "--oneline"]).decode())

# Dependencies (torch/torchvision come preinstalled with Colab's CUDA build; don't reinstall them).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODE}/requirements.txt"], check=True)

# wandb: key comes from the Colab secret WANDB_API_KEY (key icon in the left sidebar). Never hardcode it.
import wandb
wandb.login(key=userdata.get("WANDB_API_KEY"))

sys.path.insert(0, f"{CODE}/src")
# Python caches imported modules for the whole kernel session, so after a git pull
# a plain `import` would silently keep running the OLD code. Drop the cached copies.
for _m in ["fyp_infra", "stage1_runner", "stage1_analysis", "stage1_drift_diagnostic"]:
    sys.modules.pop(_m, None)
import torch
from fyp_infra import setup_paths, run_sweep, run_id, load_result
import stage1_runner as sr
paths = setup_paths()   # -> /content/drive/MyDrive/fyp_nested_cl/{data,results,logs}
print(paths)
# Hard stop for training: a CPU run is ~12x slower and its timings are useless for budgeting.
REQUIRE_GPU = True
if REQUIRE_GPU:
    assert torch.cuda.is_available(), "No GPU! Runtime -> Change runtime type -> T4 GPU, then restart."
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU runtime)")

Mounted at /content/drive
414da19 flat-llrd-reversed



/usr/local/lib/python3.13/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: ammara-asif125 (kashaf-saleem09-institute-of-business-administration) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


{'root': '/content/drive/MyDrive/fyp_nested_cl', 'data': '/content/drive/MyDrive/fyp_nested_cl/data', 'results': '/content/drive/MyDrive/fyp_nested_cl/results', 'logs': '/content/drive/MyDrive/fyp_nested_cl/logs', 'checkpoints': '/content/drive/MyDrive/fyp_nested_cl/checkpoints'}
GPU: Tesla T4


In [2]:
EPOCHS = 10            # from the epoch sweep
SEEDS = [0, 1, 2]
# Base (fast-level) lr per arm, from stage1_lr_grid. Fill these in before running.
LR = {"flat": 0.1, "flat_llrd": 0.5, "flat_llrd_rev": 0.6}   # rev: fill in from stage1_lr_grid
ARMS = {
    "flat":          dict(lr_schedule="flat"),
    "flat_llrd":     dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="slow_deep"),
    "flat_llrd_rev": dict(lr_schedule="llrd", level_ratios=(1, 4, 16), level_order="fast_deep"),
}
for name in ARMS:
    if LR[name] is None: print(f"skipping {name}: set LR['{name}'] from stage1_lr_grid first")
CONFIGS = [sr.make_cfg(stage="stage1_control", strategy="naive", epochs=EPOCHS, seed=s, lr=LR[name],
                       eval_split="test",          # final comparison -> real test set
                       save_checkpoints=True,      # weights after every task, for CKA / probes later
                       **arm)
           for s in SEEDS for name, arm in ARMS.items() if LR[name] is not None]
SHOW = ["lr", "seed", "epochs"]
for c in CONFIGS: print(run_id(c), sr.arm_label(c), {k: c[k] for k in SHOW})

4b2bd0082f5c flat {'lr': 0.1, 'seed': 0, 'epochs': 10}
f7af3b7e79aa llrd_slow_deep {'lr': 0.5, 'seed': 0, 'epochs': 10}
9d5d9fcc4194 llrd_fast_deep {'lr': 0.6, 'seed': 0, 'epochs': 10}
892486433e6d flat {'lr': 0.1, 'seed': 1, 'epochs': 10}
bf7c7b91a468 llrd_slow_deep {'lr': 0.5, 'seed': 1, 'epochs': 10}
1803d732791d llrd_fast_deep {'lr': 0.6, 'seed': 1, 'epochs': 10}
e3f9264b4607 flat {'lr': 0.1, 'seed': 2, 'epochs': 10}
bf6b7bf5e853 llrd_slow_deep {'lr': 0.5, 'seed': 2, 'epochs': 10}
b409e8563692 llrd_fast_deep {'lr': 0.6, 'seed': 2, 'epochs': 10}


In [3]:
out = run_sweep(CONFIGS, sr.run_experiment, paths)

[1/9] 4b2bd0082f5c running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 69.20%  current 69.20%  train-loss 0.909
  exp 2/20: seen-avg 43.80%  current 87.60%  train-loss 0.547
  exp 3/20: seen-avg 24.27%  current 72.80%  train-loss 0.861
  exp 4/20: seen-avg 21.95%  current 87.80%  train-loss 0.552
  exp 5/20: seen-avg 14.56%  current 72.80%  train-loss 0.861
  exp 6/20: seen-avg 13.13%  current 78.80%  train-loss 0.789
  exp 7/20: seen-avg 12.34%  current 86.40%  train-loss 0.662
  exp 8/20: seen-avg 11.00%  current 88.00%  train-loss 0.512
  exp 9/20: seen-avg 8.31%  current 74.80%  train-loss 0.606
  exp 10/20: seen-avg 8.44%  current 84.40%  train-loss 0.607
  exp 11/20: seen-avg 7.65%  current 84.20%  train-loss 0.599
  exp 12/20: seen-avg 6.42%  current 77.00%  train-loss 0.724
  exp 13/20: seen-avg 5.83%  current 75.80%  train-loss 0.771
  exp 14/20: seen-avg 6.04%  current 84.60%  train-loss 0.541
  exp 15/20: seen-avg 5.49%  current 82.40%  train-loss 0.661
  exp 16/20: seen-avg 5.42%  current 86.80%  train-loss 0.559
  exp 17/

acc_current_exp,▁▇▂▇▂▄▆▇▃▆▅▃▃▆▅▆█▆▆▇
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[2/9] f7af3b7e79aa already finished, skipping
[3/9] 9d5d9fcc4194 running


...

acc_current_exp,▃▆▅▄▆▅▄▇█▃▅▇▃▇▅▆▁▅█▅
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


[8/9] bf6b7bf5e853 already finished, skipping
[9/9] b409e8563692 running


/usr/local/lib/python3.13/dist-packages/avalanche/training/plugins/evaluation.py:94: UserWarning: No loggers specified, metrics will not be logged
  warnings.warn("No loggers specified, metrics will not be logged")


  exp 1/20: seen-avg 73.40%  current 73.40%  train-loss 0.792
  exp 2/20: seen-avg 44.10%  current 88.20%  train-loss 0.615
  exp 3/20: seen-avg 27.47%  current 82.40%  train-loss 0.608
  exp 4/20: seen-avg 18.75%  current 75.00%  train-loss 0.716
  exp 5/20: seen-avg 16.00%  current 80.00%  train-loss 0.528
  exp 6/20: seen-avg 13.67%  current 82.00%  train-loss 0.602
  exp 7/20: seen-avg 11.94%  current 83.60%  train-loss 0.509
  exp 8/20: seen-avg 10.57%  current 84.60%  train-loss 0.540
  exp 9/20: seen-avg 10.20%  current 91.80%  train-loss 0.371
  exp 10/20: seen-avg 8.14%  current 81.40%  train-loss 0.678
  exp 11/20: seen-avg 8.00%  current 88.00%  train-loss 0.488
  exp 12/20: seen-avg 7.52%  current 90.20%  train-loss 0.368
  exp 13/20: seen-avg 5.52%  current 71.80%  train-loss 0.765
  exp 14/20: seen-avg 6.27%  current 87.80%  train-loss 0.472
  exp 15/20: seen-avg 5.84%  current 87.60%  train-loss 0.552
  exp 16/20: seen-avg 5.19%  current 83.00%  train-loss 0.516
  exp 17

acc_current_exp,▂▇▅▂▄▅▅▅█▄▇▇▁▇▇▅▂▅▆▄
acc_exp/00,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/01,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/02,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/03,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/04,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/05,█▁▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/06,█▁▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/07,█▁▁▁▁▁▁▁▁▁▁▁▁
acc_exp/08,█▁▁▁▁▁▁▁▁▁▁▁
+15,...


In [4]:
import numpy as np, pandas as pd
rows = []
for cfg in CONFIGS:
    r = load_result(cfg, paths)
    if r is None:
        rows.append({**{k: cfg[k] for k in SHOW}, "status": "not finished"}); continue
    m = r["metrics"]
    rows.append({"arm": sr.arm_label(cfg), **{k: cfg[k] for k in SHOW}, "run_id": r["run_id"],
                 "final_avg_acc": m["final_avg_acc"], "avg_forgetting": m["avg_forgetting"],
                 "avg_incremental_acc": m["avg_incremental_acc"],
                 # mean of the diagonal: how well each task was learned when it was current.
                 # Report next to forgetting: forgetting = peak - final, so it grows with this.
                 "acc_just_learned": float(np.mean(np.diag(m["acc_matrix"]))),
                 "steps": m["optimizer_steps"], "n_params": m["n_params"], "minutes": r["wall_clock_s"] / 60})
df = pd.DataFrame(rows); df

,arm,lr,seed,epochs,run_id,final_avg_acc,avg_forgetting,avg_incremental_acc,acc_just_learned,steps,n_params,minutes
0,flat,0.1,0,10,4b2bd0082f5c,4.40,81.936842,14.147757,82.24,15800,1109240,9.435934
1,llrd_slow_deep,0.5,0,10,f7af3b7e79aa,4.26,81.789474,13.578575,81.96,15800,1109240,9.977412
2,llrd_fast_deep,0.6,0,10,9d5d9fcc4194,4.47,81.568421,14.047487,81.96,15800,1109240,8.561422
3,flat,0.1,1,10,892486433e6d,4.49,82.136842,13.529266,82.52,15800,1109240,8.624697
4,llrd_slow_deep,0.5,1,10,bf7c7b91a468,4.46,82.084211,13.847335,82.44,15800,1109240,10.034140
5,llrd_fast_deep,0.6,1,10,1803d732791d,4.47,81.494737,13.804185,81.89,15800,1109240,9.088601
6,flat,0.1,2,10,e3f9264b4607,4.15,82.305263,14.553691,82.34,15800,1109240,8.551136
7,llrd_slow_deep,0.5,2,10,bf6b7bf5e853,3.90,81.452632,14.257252,81.28,15800,1109240,9.822285
8,llrd_fast_deep,0.6,2,10,b409e8563692,4.06,82.947368,14.512349,82.86,15800,1109240,9.638603


In [5]:
done = df[df.get("status") != "not finished"] if "status" in df else df
assert done["steps"].nunique() == 1 and done["n_params"].nunique() == 1, "steps or param count differ between arms. The comparison is NOT matched"
print(done.groupby(["arm", "lr"])[["final_avg_acc", "avg_forgetting", "avg_incremental_acc"]].agg(["mean", "std", "count"]))
print(done.groupby(["arm", "lr"])["acc_just_learned"].agg(["mean", "std"]))

                   final_avg_acc                 avg_forgetting            \
                            mean       std count           mean       std   
arm            lr                                                           
flat           0.1      4.346667  0.176163     3      82.126316  0.184436   
llrd_fast_deep 0.6      4.333333  0.236714     3      82.003509  0.818236   
llrd_slow_deep 0.5      4.206667  0.283784     3      81.775439  0.316023   

                         avg_incremental_acc                  
                   count                mean       std count  
arm            lr                                             
flat           0.1     3           14.076904  0.515875     3  
llrd_fast_deep 0.6     3           14.121340  0.359812     3  
llrd_slow_deep 0.5     3           13.894387  0.341776     3  
                         mean       std
arm            lr                      
flat           0.1  82.366667  0.141892
llrd_fast_deep 0.6  82.236667  0.540956